# Custodian: isolated CICIDS2017 training workflow

This notebook is designed for a **temporary Google Colab runtime**. It trains the existing Custodian Behaviour model using only the three approved CICIDS2017 CSV files, reusing Custodian's shared feature engine.

**Safety contract:** no local-network access, no packet capture, no traffic replay, no payload execution, no credentials/private files, and no dataset/model artifacts are committed to Git.


## 0. Runtime safety checklist

Before continuing:
- Runtime is temporary/isolated.
- Do not connect this notebook/runtime to a local network or private VPN.
- Do not upload credentials, private files, or unrelated personal data.
- Use only the approved CICIDS2017 CSV flow files listed below.
- Do not use live packet capture or replay traffic.
- Do not execute files extracted from packet payloads.

The training code itself requires an explicit acknowledgement plus `CUSTODIAN_ISOLATED_TRAINING=YES` before model fitting.

In [ ]:
# Pin the environment used by this workflow. Do not replace these with unpinned installs.
%pip install -q --upgrade pip
%pip install -q "numpy==2.2.6" "pandas==2.2.3" "pyarrow==21.0.0" "scikit-learn==1.7.2" "xgboost==3.0.2" "joblib==1.5.2" "pytest==8.4.2" "ruff==0.13.0" "fastapi==0.116.1" "uvicorn==0.35.0" "pydantic==2.11.7" "pydantic-settings==2.10.1" "PyYAML==6.0.2" "dpkt==1.9.8" "psutil==7.0.0" "websockets==15.0.1"

In [ ]:
import os, sys, subprocess, importlib.metadata as md
from pathlib import Path

PINNED = {
    "numpy": "2.2.6", "pandas": "2.2.3", "pyarrow": "21.0.0",
    "scikit-learn": "1.7.2", "xgboost": "3.0.2", "joblib": "1.5.2",
    "pytest": "8.4.2", "ruff": "0.13.0", "fastapi": "0.116.1",
    "uvicorn": "0.35.0", "pydantic": "2.11.7", "pydantic-settings": "2.10.1",
    "PyYAML": "6.0.2", "dpkt": "1.9.8", "psutil": "7.0.0", "websockets": "15.0.1",
}
for package, expected in PINNED.items():
    actual = md.version(package)
    assert actual == expected, f"{package}: expected {expected}, got {actual}"
print("All pinned dependency versions verified.")
print("Python:", sys.version.split()[0])

In [ ]:
# Clone the public repository at the revision being tested.
import shutil, subprocess
REPO = Path('/content/Custodian')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/EmberFalls/Custodian.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
print('Repository:', REPO)


In [ ]:
# Upload ONLY the three approved CSVs. They stay in the temporary Colab runtime.
from google.colab import files
import hashlib, shutil

APPROVED = {
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
}
uploaded = files.upload()
unexpected = sorted(set(uploaded) - APPROVED)
missing = sorted(APPROVED - set(uploaded))
if unexpected or missing:
    raise ValueError(f'Approved-file check failed. Unexpected={unexpected}; missing={missing}')

DATA_DIR = REPO / 'data' / 'raw' / 'cicids2017'
DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in sorted(APPROVED):
    src = Path('/content') / name
    dst = DATA_DIR / name
    shutil.copy2(src, dst)
    digest = hashlib.sha256(dst.read_bytes()).hexdigest()
    print(f'{name}: {dst.stat().st_size:,} bytes | sha256={digest}')


In [ ]:
# Verify the repository's own source validation and inspect labels/schema before training.
from training.cicids2017 import REQUIRED_FILES, normalize_columns
import pandas as pd

for name in REQUIRED_FILES:
    path = DATA_DIR / name
    header = pd.read_csv(path, nrows=0)
    normalize_columns(header)
    print(f'OK schema: {name}')
    labels = pd.read_csv(path, usecols=['Label'])['Label'].astype(str).str.strip()
    print(labels.value_counts().to_string())


In [ ]:
# Set the explicit safety gate required by Custodian's training module.
os.environ['CUSTODIAN_ISOLATED_TRAINING'] = 'YES'
print('Isolation acknowledgement set for this temporary runtime only.')


In [ ]:
# Prepare data and train exactly one Behaviour model.
# The source code refuses to overwrite an existing artifact directory.
import subprocess

MODEL_DIR = REPO / 'model_artifacts' / 'behaviour-xgb-colab-v1'
cmd = [
    sys.executable, '-m', 'training.train_behaviour',
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(MODEL_DIR),
    '--n-jobs', '2',
    '--acknowledge-isolated-vm',
]
subprocess.run(cmd, check=True)
print('Model package created at', MODEL_DIR)

In [ ]:
# Reproducibility / integrity checks.
import json, hashlib

required_artifacts = ['model.json','calibrator.joblib','feature_schema.json','class_mapping.json','thresholds.json','metrics.json','manifest.json']
for name in required_artifacts:
    assert (MODEL_DIR / name).is_file(), f'Missing artifact: {name}'
manifest = json.loads((MODEL_DIR / 'manifest.json').read_text())
for name, expected in manifest['artifact_sha256'].items():
    actual = hashlib.sha256((MODEL_DIR / name).read_bytes()).hexdigest()
    assert actual == expected, f'Checksum mismatch: {name}'
metrics = json.loads((MODEL_DIR / 'metrics.json').read_text())
print(json.dumps(metrics['final_test_calibrated'], indent=2))
print('Artifact checksums verified.')

In [ ]:
# Export a versioned package for inspection/download. This ZIP is NOT committed to Git.
import shutil
package = shutil.make_archive('/content/custodian-behaviour-xgb-colab-v1', 'zip', MODEL_DIR)
metrics_path = MODEL_DIR / 'metrics.json'
manifest_path = MODEL_DIR / 'manifest.json'
print('Model package:', package)
print('Metrics:', metrics_path)
print('Manifest:', manifest_path)

from google.colab import files as colab_files
colab_files.download(package)

In [ ]:
# Clean up uploaded data and generated artifacts from the temporary runtime.
import shutil
for path in [DATA_DIR, REPO / 'data' / 'processed', REPO / 'data' / 'manifests', MODEL_DIR]:
    if path.exists():
        shutil.rmtree(path)
for name in APPROVED:
    path = Path('/content') / name
    if path.exists():
        path.unlink()
print('Temporary dataset and model artifacts removed from the Colab runtime.')